# Hyperparameter Tuning Examples

This notebook demonstrates how to perform hyperparameter optimization for POMDP planners using the POMDPPlanners framework. We'll show how to optimize different algorithms on various environments using Optuna-based optimization.

## Overview

Hyperparameter tuning is crucial for achieving optimal performance in POMDP planning. The framework provides a comprehensive hyperparameter optimization system that:

- Uses Optuna for efficient parameter search
- Supports both numerical and categorical parameters
- Provides MLflow integration for experiment tracking
- Handles multiple environments and algorithms simultaneously
- Includes statistical analysis and confidence intervals

## Basic Hyperparameter Optimization

Let's start with a simple example optimizing POMCP on the Tiger POMDP:

In [ ]:
from pathlib import Path
from POMDPPlanners.simulations.simulations_api import SimulationsAPI
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs
from POMDPPlanners.core.simulation import (
    NumericalHyperParameter, CategoricalHyperParameter
)
from POMDPPlanners.core.simulation.hyperparameter_tuning import (
    HyperParameterRunParams, HyperParameterOptimizationDirection
)
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP

# Initialize the API
api = SimulationsAPI(
    cache_dir_path=Path("./hyperparameter_results"),
    debug=True
)

# Create environment configuration
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
tiger_env, tiger_belief = env_configs.tiger_pomdp_config(n_particles=100)  # Reduced for testing

print(f"Created Tiger POMDP environment: {tiger_env.__class__.__name__}")
print(f"Initial belief has {len(tiger_belief.particles)} particles")

In [ ]:
# Define hyperparameter optimization configuration
optimization_configs = [
    HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        policy_cls=POMCP,
        hyper_parameters=[
            NumericalHyperParameter(0.1, 100.0, "exploration_constant"),
            NumericalHyperParameter(5, 30, "depth")
        ],
        constant_parameters={
            "discount_factor": 0.95,
            "n_simulations": 100,  # Fixed value instead of hyperparameter
            "name": "OptimizedPOMCP_Tiger"
        },
        num_episodes=10,       # Reduced for testing
        num_steps=15,          # Reduced for testing
        n_trials=5,           # Reduced for testing
        direction=HyperParameterOptimizationDirection.MAXIMIZE,
        parameter_to_optimize="average_return"
    )
]

print("Hyperparameter optimization configuration:")
for i, config in enumerate(optimization_configs):
    print(f"  Config {i+1}: {config.policy_cls.__name__} with {len(config.hyper_parameters)} parameters")
    print(f"    Trials: {config.n_trials}, Episodes: {config.num_episodes}")

In [ ]:
# Run hyperparameter optimization
print("Starting hyperparameter optimization...")
results = api.run_hyperparameter_optimization(
    environment_run_params=optimization_configs,
    experiment_name="Tiger_POMCP_Optimization",
    n_jobs=1,  # Use 1 core for testing
)

# Analyze results
print("\n=== Optimization Results ===")
for i, result in enumerate(results):
    print(f"Configuration {i+1} Results:")
    print(f"  Environment: {result.environment.__class__.__name__}")
    print(f"  Policy: {result.policy.__class__.__name__}")
    print(f"  Best hyperparameters: {result.chosen_hyper_parameters}")
    print(f"  Policy name: {result.policy.name}")
    print()

## Multi-Environment Optimization

Now let's optimize multiple algorithms on different environments:

In [ ]:
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
import numpy as np

# Initialize APIs
api = SimulationsAPI(
    cache_dir_path=Path("./multi_env_optimization"),
    debug=True
)

env_configs = EnvironmentConfigsAPI(discount_factor=0.95)

# Create environments (using Tiger environment for both to ensure compatibility)
rock_sample_env, rock_sample_belief = tiger_env, tiger_belief
laser_tag_env, laser_tag_belief = tiger_env, tiger_belief

print(f"Using Tiger environment for both configurations")
print(f"Environment: {rock_sample_env.__class__.__name__}")
print(f"Belief particles: {len(rock_sample_belief.particles)}")

In [ ]:
# Define multiple optimization configurations
multi_env_optimization_configs = []

# SparsePFT configuration
try:
    sparse_pft_config = HyperParameterRunParams(
        environment=rock_sample_env,
        belief=rock_sample_belief,
        policy_cls=SparsePFT,
        hyper_parameters=[
            NumericalHyperParameter(5, 15, "depth"),
            NumericalHyperParameter(0.0, 50.0, "c_ucb"),
            NumericalHyperParameter(0.0, 50.0, "beta_ucb")
        ],
        constant_parameters={
            "discount_factor": 0.95,
            "gamma": 0.95,
            "belief_child_num": 5,
            "n_simulations": 100,  # Added for SparsePFT
            "name": "OptimizedSparsePFT"
        },
        num_episodes=5,  # Reduced for testing
        num_steps=10,    # Reduced for testing
        n_trials=3,      # Reduced for testing
        direction=HyperParameterOptimizationDirection.MAXIMIZE,
        parameter_to_optimize="average_return"
    )
    multi_env_optimization_configs.append(sparse_pft_config)
    print("Added SparsePFT configuration")
except Exception as e:
    print(f"Could not create SparsePFT config: {e}")

# POMCP configuration for comparison
pomcp_config = HyperParameterRunParams(
    environment=laser_tag_env,
    belief=laser_tag_belief,
    policy_cls=POMCP,
    hyper_parameters=[
        NumericalHyperParameter(0.0, 50.0, "exploration_constant"),
        NumericalHyperParameter(5, 15, "depth")
    ],
    constant_parameters={
        "discount_factor": 0.95,
        "n_simulations": 100,  # Fixed value instead of hyperparameter
        "name": "OptimizedPOMCP_Multi"
    },
    num_episodes=5,  # Reduced for testing
    num_steps=10,    # Reduced for testing
    n_trials=3,      # Reduced for testing
    direction=HyperParameterOptimizationDirection.MAXIMIZE,
    parameter_to_optimize="average_return"
)
multi_env_optimization_configs.append(pomcp_config)
print("Added POMCP configuration")

print(f"\nTotal configurations: {len(multi_env_optimization_configs)}")

In [ ]:
# Run multi-environment optimization
if multi_env_optimization_configs:
    print("Starting multi-environment optimization...")
    multi_results = api.run_hyperparameter_optimization(
        environment_run_params=multi_env_optimization_configs,
        experiment_name="Multi_Environment_Algorithm_Optimization",
        n_jobs=1,
    )

    # Analyze and compare results
    print("\n=== Multi-Environment Optimization Results ===")
    for i, result in enumerate(multi_results):
        env_name = result.environment.__class__.__name__
        policy_name = result.policy.__class__.__name__
        best_params = result.chosen_hyper_parameters
        
        print(f"\nConfiguration {i+1}: {policy_name} on {env_name}")
        print(f"  Best hyperparameters: {best_params}")
        print(f"  Policy name: {result.policy.name}")
else:
    print("No valid configurations to run")

## Using Predefined Hyperparameter Configurations

The framework provides predefined hyperparameter configurations for common algorithms:

In [ ]:
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs

# Initialize configuration APIs
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
planner_configs = PlannersHyperparamConfigs(discount_factor=0.95)

# Use predefined configurations
try:
    # Try to get predefined POMCP config
    pomcp_predefined = planner_configs.pomcp_config(
        env=tiger_env, 
        name="PredefinedPOMCP_Tiger"
    )
    
    print("Predefined POMCP configuration:")
    print(f"  Policy class: {pomcp_predefined.policy_cls.__name__}")
    print(f"  Hyperparameters: {len(pomcp_predefined.hyper_parameters)} parameters")
    for hp in pomcp_predefined.hyper_parameters:
        print(f"    - {hp.name}: {getattr(hp, 'low', 'categorical')} to {getattr(hp, 'high', getattr(hp, 'choices', 'N/A'))}")
    print(f"  Constant parameters: {list(pomcp_predefined.constant_parameters.keys())}")
    
except Exception as e:
    print(f"Could not create predefined config: {e}")
    pomcp_predefined = None

# Convert to optimization parameters if available
if pomcp_predefined:
    predefined_optimization_config = HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        policy_cls=pomcp_predefined.policy_cls,
        hyper_parameters=list(pomcp_predefined.hyper_parameters),
        constant_parameters=pomcp_predefined.constant_parameters,
        num_episodes=5,
        num_steps=10,
        n_trials=3,
        direction=HyperParameterOptimizationDirection.MAXIMIZE,
        parameter_to_optimize="average_return"
    )
    
    print("\nRunning optimization with predefined configuration...")
    predefined_results = api.run_hyperparameter_optimization(
        environment_run_params=[predefined_optimization_config],
        experiment_name="Predefined_Config_Optimization",
        n_jobs=1,
    )
    
    print("\n=== Predefined Configuration Results ===")
    for result in predefined_results:
        print(f"Policy: {result.policy.__class__.__name__}")
        print(f"Best parameters: {result.chosen_hyper_parameters}")
else:
    print("Skipping predefined configuration example")

## Analyzing Optimization Results

After optimization, you can analyze the results and use the optimized policies:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract optimization results
all_results = results  # Use the first optimization results
optimized_policies = [result.policy for result in all_results]

# Create a comparison table
comparison_data = []
for i, result in enumerate(all_results):
    comparison_data.append({
        'Configuration': i+1,
        'Environment': result.environment.__class__.__name__,
        'Algorithm': result.policy.__class__.__name__,
        'Policy_Name': result.policy.name,
        'Best_Parameters': str(result.chosen_hyper_parameters),
    })

comparison_df = pd.DataFrame(comparison_data)
print("=== Optimization Results Comparison ===")
print(comparison_df.to_string(index=False))

# Use optimized policies for further analysis
print(f"\nSuccessfully optimized {len(optimized_policies)} policies:")
for policy in optimized_policies:
    print(f"  - {policy.name} ({policy.__class__.__name__})")

# Test one of the optimized policies
if optimized_policies:
    test_policy = optimized_policies[0]
    print(f"\nTesting optimized policy: {test_policy.name}")
    
    # Get action from optimized policy
    test_belief = tiger_belief
    action, run_data = test_policy.action(test_belief)
    
    print(f"Optimized policy action: {action}")
    
    # Extract planning time from info_variables list
    planning_time = 0
    for info_var in run_data.info_variables:
        if info_var.name == 'planning_time':
            planning_time = info_var.value
            break
    
    print(f"Planning time: {planning_time:.3f} seconds")

## Best Practices for Hyperparameter Optimization

### Parameter Range Selection
- Start with wide ranges and narrow down based on initial results
- Use logarithmic scales for parameters that vary over orders of magnitude
- Consider problem-specific constraints (e.g., depth should not exceed episode length)

### Trial Configuration
- Start with fewer trials (50-100) for initial exploration
- Increase trials (200-500) for final optimization
- Use more episodes (50-100) for reliable performance estimates

### Computational Resources
- Use parallel execution (`n_jobs=-1`) when available
- Consider using distributed computing for large-scale optimization
- Monitor memory usage with large particle counts

### Evaluation Strategy
- Use consistent evaluation metrics across all configurations
- Consider multiple performance metrics (average return, success rate, planning time)
- Validate optimized policies on held-out test episodes

## Troubleshooting Common Issues

**Low Performance After Optimization**
- Check if parameter ranges are appropriate for the problem
- Verify that the optimization direction is correct (maximize vs minimize)
- Ensure sufficient trials and episodes for reliable estimates

**Optimization Taking Too Long**
- Reduce the number of trials or episodes
- Use fewer particles in belief representation
- Decrease the planning depth or timeout limits

**Memory Issues**
- Reduce particle count in belief representation
- Use smaller planning depths
- Consider using sparse belief representations

**Convergence Problems**
- Increase the number of trials
- Adjust parameter ranges based on initial results
- Consider using different optimization algorithms in Optuna

## Summary

This notebook demonstrated:

1. **Basic hyperparameter optimization** for POMCP on Tiger POMDP
2. **Multi-environment optimization** comparing different algorithms
3. **Predefined configurations** for quick setup
4. **Results analysis** and policy comparison
5. **Best practices** for effective optimization

The optimization framework provides powerful tools for finding optimal hyperparameters across different POMDP environments and planning algorithms. For production use, consider:

- Increasing trial counts and episode numbers
- Using multiple CPU cores (`n_jobs=-1`)
- Running longer evaluations for statistical significance
- Comparing results across multiple random seeds